# Notebook agent: EDA ➜ training notebook

This agent uses **Gemini** to read `EDA.ipynb` (the Oxford-IIIT Pet segmentation EDA) from your Google Drive
and write a **separate training notebook** based on what the EDA found. It saves that notebook into your Drive folder.

**What it does**
1. **Read**: downloads `EDA.ipynb` through the Drive API and condenses its markdown, code and text outputs into context.
2. **Plan**: asks Gemini for a section-by-section plan for a training notebook based on the EDA's findings.
3. **Write**: asks Gemini for the notebook cells (structured JSON output).
4. **Check & repair**: parses every code cell, runs a checklist review, and sends problems back to Gemini until they are fixed (max `MAX_REPAIR_ROUNDS`).
5. **Save**: builds a valid `.ipynb` file and uploads it to the target Drive folder, then prints the Colab link.

**One-time setup:** in Colab, open 🔑 **Secrets** (left sidebar). Add `GOOGLE_API_KEY` with a key from
[Google AI Studio](https://aistudio.google.com/apikey) and turn on *Notebook access*. Then choose **Runtime ▸ Run all**.

In [ ]:
!pip install -q -U google-genai nbformat

## 1. Configuration

In [ ]:
# Drive IDs (taken from the share links)
SOURCE_NOTEBOOK_ID = "1okZfBMyHmqpOJmmSzBrMWy7-ZBOaYPLC"   # EDA.ipynb
TARGET_FOLDER_ID   = "1taLH7ezsguMihBwcq95m_G4M_ZVucWal"   # where the new notebook is saved

OUTPUT_NAME = "Pet_Segmentation_Training.ipynb"   # a timestamp is added if the name already exists in the folder

# Tried in order; the first model your key can use wins
MODEL_CANDIDATES = ["gemini-3-pro-preview", "gemini-2.5-pro", "gemini-2.5-flash"]
MAX_REPAIR_ROUNDS = 3

TASK = """
Write a complete, runnable Google Colab notebook that trains a semantic segmentation model on the
Oxford-IIIT Pet dataset, building directly on the EDA notebook provided as context.

Requirements:
- Setup: install/import what is needed, pick GPU if available, set seeds (42) for reproducibility.
- Data: expect the dataset at /content/dataset (images/*.jpg, annotations/trimaps/*.png, annotations/trainval.txt, test.txt).
  If it is missing, download and extract the official archives
  (https://thor.robots.ox.ac.uk/datasets/pets/images.tar.gz and annotations.tar.gz) into /content/dataset.
- Splits: prefer the clean lists written by the EDA in /content/dataset/clean_splits/{train,val,test}.txt
  (one filename per line, no header) and manifest.csv. If they don't exist, fall back to the official
  trainval/test lists with an 80/20 breed-stratified train/val split (random_state=42), and drop masks with no pet pixels.
  Skip macOS "._" files.
- Labels: trimap values are 1=pet, 2=background, 3=boundary. Remap to 0..2 and make the boundary handling
  explicit and configurable (train it as its own class, or ignore_index it in the loss).
- Model: a U-Net-style encoder-decoder with a pretrained ImageNet encoder (e.g. segmentation_models_pytorch or torchvision),
  input size configurable (default 256), mixed precision when on GPU.
- Imbalance: the EDA shows that background dominates and that foreground share varies widely, so use
  class-weighted cross-entropy + Dice loss with weights computed from the training masks.
- Augmentation: flips, scale/crop and colour jitter applied consistently to image and mask (nearest-neighbour for masks).
  Keep in mind that pets are centre-biased and often touch the border.
- Training: train/val loop with a progress bar, LR scheduler, early stopping, best-checkpoint saving to /content/checkpoints.
- Evaluation: per-class IoU and mean IoU, pixel accuracy, plus a per-breed / cats-vs-dogs IoU breakdown on the test set.
  Report boundary-band metrics separately from the pet/background IoU.
- Visuals: training curves, and image | ground truth | prediction grids for best and worst test samples.
- Structure: clear markdown headings and short explanations that reference the EDA findings that drove each choice.
- Keep a CONFIG dict at the top (epochs, batch size, image size, lr, boundary mode) so the notebook is easy to tweak.
"""

## 2. Google Drive access

In [ ]:
import io, json
from google.colab import auth, userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload

auth.authenticate_user()
drive = build("drive", "v3")


def drive_download_text(file_id):
    buf = io.BytesIO()
    req = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    dl = MediaIoBaseDownload(buf, req)
    done = False
    while not done:
        _, done = dl.next_chunk()
    return buf.getvalue().decode("utf-8")


def drive_name_taken(name, folder_id):
    q = f"name = '{name}' and '{folder_id}' in parents and trashed = false"
    res = drive.files().list(q=q, fields="files(id)", supportsAllDrives=True,
                             includeItemsFromAllDrives=True).execute()
    return bool(res.get("files"))


def drive_upload_notebook(nb_json, name, folder_id):
    media = MediaIoBaseUpload(io.BytesIO(nb_json.encode("utf-8")), mimetype="application/x-ipynb+json")
    meta = {"name": name, "parents": [folder_id]}
    return drive.files().create(body=meta, media_body=media, fields="id,name,webViewLink",
                                supportsAllDrives=True).execute()


src_meta = drive.files().get(fileId=SOURCE_NOTEBOOK_ID, fields="name", supportsAllDrives=True).execute()
dst_meta = drive.files().get(fileId=TARGET_FOLDER_ID, fields="name", supportsAllDrives=True).execute()
print(f"Source notebook: {src_meta['name']}  |  Target folder: {dst_meta['name']}")

## 3. Read the EDA notebook into context

In [ ]:
import nbformat

MAX_OUTPUT_CHARS = 1500   # per cell: enough to keep printed findings, but no image blobs


def notebook_to_context(nb):
    parts = []
    for i, cell in enumerate(nb.cells):
        if cell.cell_type == "markdown":
            parts.append(f"### [cell {i} | markdown]\n{cell.source}")
        elif cell.cell_type == "code":
            parts.append(f"### [cell {i} | code]\n```python\n{cell.source}\n```")
            texts = []
            for out in cell.get("outputs", []):
                if out.get("output_type") == "stream":
                    texts.append(out.get("text", ""))
                elif out.get("output_type") in ("execute_result", "display_data"):
                    texts.append(out.get("data", {}).get("text/plain", ""))
                elif out.get("output_type") == "error":
                    texts.append(f"{out.get('ename')}: {out.get('evalue')}")
            text = "".join(t if isinstance(t, str) else "".join(t) for t in texts).strip()
            if text:
                parts.append(f"[output]\n{text[:MAX_OUTPUT_CHARS]}")
    return "\n\n".join(parts)


eda_nb = nbformat.reads(drive_download_text(SOURCE_NOTEBOOK_ID), as_version=4)
EDA_CONTEXT = notebook_to_context(eda_nb)
print(f"{len(eda_nb.cells)} cells -> {len(EDA_CONTEXT):,} characters of context")

## 4. Gemini client

In [ ]:
from typing import Literal
from pydantic import BaseModel
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))


class Cell(BaseModel):
    cell_type: Literal["markdown", "code"]
    source: str


class NotebookDraft(BaseModel):
    title: str
    cells: list[Cell]


class Review(BaseModel):
    ok: bool
    problems: list[str]


SYSTEM = (
    "You are a senior ML engineer who writes clean, correct, runnable Google Colab notebooks. "
    "Code must run top-to-bottom on a fresh Colab runtime. Use '!pip install' for missing packages. "
    "Never invent file paths or column names that the EDA context does not support."
)

MODEL = None


def ask(prompt, schema=None, temperature=0.3):
    """Calls Gemini, falling back through MODEL_CANDIDATES until one works."""
    global MODEL
    cfg = types.GenerateContentConfig(
        system_instruction=SYSTEM, temperature=temperature, max_output_tokens=65536,
        **({"response_mime_type": "application/json", "response_schema": schema} if schema else {}))
    for m in ([MODEL] if MODEL else MODEL_CANDIDATES):
        try:
            resp = client.models.generate_content(model=m, contents=prompt, config=cfg)
            MODEL = m
            return resp.parsed if schema else resp.text
        except Exception as e:
            if MODEL:
                raise
            print(f"  model {m} unavailable ({type(e).__name__}: {str(e)[:120]}), trying next")
    raise RuntimeError("None of MODEL_CANDIDATES worked with this API key")

## 5. Run the agent: plan ➜ write ➜ check & repair

In [ ]:
import ast, re


def static_problems(draft):
    """Syntax-checks every code cell (ignoring ! shell and % magic lines)."""
    problems = []
    for i, c in enumerate(draft.cells):
        if c.cell_type != "code":
            continue
        body = "\n".join("pass" if l.lstrip().startswith(("!", "%")) else l for l in c.source.splitlines())
        try:
            ast.parse(body)
        except SyntaxError as e:
            problems.append(f"cell {i}: SyntaxError line {e.lineno}: {e.msg}")
    if not any(c.cell_type == "code" for c in draft.cells):
        problems.append("notebook has no code cells")
    return problems


def draft_to_text(draft):
    return "\n\n".join(f"### [cell {i} | {c.cell_type}]\n{c.source}" for i, c in enumerate(draft.cells))


print("Step 1/3 - planning...")
plan = ask(f"""TASK:\n{TASK}\n\nEDA NOTEBOOK (context):\n{EDA_CONTEXT}\n\n
Write a concise section-by-section plan for the new notebook. For each section list the cells
and state which EDA finding (quote the numbers) motivates the design choice.""")
print(f"  using model: {MODEL}\n")
print(plan[:3000])

print("\nStep 2/3 - writing the notebook...")
draft = ask(f"""TASK:\n{TASK}\n\nEDA NOTEBOOK (context):\n{EDA_CONTEXT}\n\nPLAN:\n{plan}\n\n
Now write the full notebook following the plan. Return every cell in order. Code cells must contain
complete code, with no placeholders, ellipses or 'TODO'.""", schema=NotebookDraft)
print(f"  drafted {len(draft.cells)} cells: '{draft.title}'")

print("\nStep 3/3 - checking & repairing...")
for rnd in range(1, MAX_REPAIR_ROUNDS + 1):
    problems = static_problems(draft)
    review = ask(f"""TASK:\n{TASK}\n\nNOTEBOOK DRAFT:\n{draft_to_text(draft)}\n\n
Review the draft strictly against every TASK requirement and for bugs that would crash at runtime
(undefined names, wrong tensor shapes/dtypes, mask label mapping, mismatched paths, missing imports,
cells that depend on variables defined later). List only concrete problems.""", schema=Review, temperature=0)
    problems += review.problems
    print(f"  round {rnd}: {len(problems)} problem(s)")
    for p in problems:
        print("   -", p[:200])
    if not problems:
        break
    draft = ask(f"""TASK:\n{TASK}\n\nEDA NOTEBOOK (context):\n{EDA_CONTEXT}\n\nNOTEBOOK DRAFT:\n{draft_to_text(draft)}\n\n
Fix ALL of these problems and return the complete corrected notebook (all cells):\n- """ + "\n- ".join(problems),
                 schema=NotebookDraft)
else:
    print("  stopped after MAX_REPAIR_ROUNDS; remaining issues are listed above")

## 6. Build the notebook and save it to Drive

In [ ]:
from datetime import datetime

for p in static_problems(draft):
    print("⚠️ still unresolved:", p)

nb =nbformat.v4.new_notebook()
nb.metadata = {
    "kernelspec": {"name": "python3", "display_name": "Python 3"},
    "language_info": {"name": "python"},
    "accelerator": "GPU",
    "colab": {"provenance": [], "gpuType": "T4"},
}
nb.cells.append(nbformat.v4.new_markdown_cell(
    f"*Generated by the notebook agent ({MODEL}) from `{src_meta['name']}` on {datetime.now():%Y-%m-%d %H:%M}.*"))
for c in draft.cells:
    make = nbformat.v4.new_code_cell if c.cell_type == "code" else nbformat.v4.new_markdown_cell
    nb.cells.append(make(c.source))
nbformat.validate(nb)

name = OUTPUT_NAME
if drive_name_taken(name, TARGET_FOLDER_ID):
    name = name.replace(".ipynb", f"_{datetime.now():%Y%m%d_%H%M%S}.ipynb")

f = drive_upload_notebook(nbformat.writes(nb), name, TARGET_FOLDER_ID)
print(f"✅ Saved '{f['name']}' to Drive folder '{dst_meta['name']}'")
print(f"Open in Colab: https://colab.research.google.com/drive/{f['id']}")